# 04 — Land Use Mix (HCAD)

Computes Shannon entropy of HCAD land use distribution per grid cell to measure how mixed vs. homogeneous each cell is.

**Data source:** HCAD consolidated CSV (`hcad_path`).

**Note:** Uses HCAD `landuse` codes and lot area for area-weighted entropy calculation.

**Output columns:** `cell_id`, `landuse_entropy`

**Output file:** `csv/Houston/04_land_use_mix.csv`

In [5]:
# ── Papermill parameters ──────────────────────────────
GRID_CONFIG = "grid.json"

In [6]:
import pandas as pd
import numpy as np
import json
import os
import math

with open(GRID_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

HCAD_PATH = config["hcad_path"]
CELL_SIZE_M = config["grid_cell_size_m"]
CSV_DIR = config.get("csv_dir", "csv")

os.makedirs(CSV_DIR, exist_ok=True)

if not config["feature_flags"].get("needs_hcad", False):
    print("HCAD not available — skipping notebook 04.")
    df_grid = pd.read_csv(f"{CSV_DIR}/01_grid_definition.csv", dtype={"cell_id": str})
    df_empty = pd.DataFrame({"cell_id": df_grid["cell_id"]})
    df_empty["landuse_entropy"] = np.nan
    df_empty.to_csv(f"{CSV_DIR}/04_land_use_mix.csv", index=False)
    raise SystemExit("Skipped — needs_hcad=false")

print(f"Loading HCAD from {HCAD_PATH}")

Loading HCAD from hcad/harris_county_building_data.csv


In [10]:
# ── Load HCAD + Grid definition ──────────────────────
COLS = ["latitude", "longitude", "landuse", "lotarea"]

df_hcad = pd.read_csv(HCAD_PATH, usecols=COLS, dtype={"landuse": str})
df_hcad["lotarea"] = pd.to_numeric(df_hcad["lotarea"], errors="coerce").fillna(0)
df_hcad["latitude"] = pd.to_numeric(df_hcad["latitude"], errors="coerce")
df_hcad["longitude"] = pd.to_numeric(df_hcad["longitude"], errors="coerce")
df_hcad = df_hcad.dropna(subset=["latitude", "longitude"]).copy()

print(f"Loaded {len(df_hcad):,} HCAD records")

# Load grid definition from notebook 01 (has the authoritative cell_id values)
df_grid = pd.read_csv(f"{CSV_DIR}/01_grid_definition.csv", dtype={"cell_id": str})
print(f"Loaded {len(df_grid):,} grid cells")

# Use BallTree to assign HCAD records to nearest grid cells
# (This matches notebook 01's spatial assignment method)
from sklearn.neighbors import BallTree

coords = df_grid[["cell_lat", "cell_lon"]].values
tree = BallTree(np.radians(coords), metric="haversine")

hcad_coords = df_hcad[["latitude", "longitude"]].values
distances, indices = tree.query(np.radians(hcad_coords), k=1)

df_hcad["cell_id"] = df_grid.iloc[indices.flatten()]["cell_id"].values

print(f"Assigned {len(df_hcad):,} records to grid cells")
print(f"Cells with data: {df_hcad['cell_id'].nunique():,}")

Loaded 1,592,879 HCAD records
Loaded 54,947 grid cells
Assigned 1,592,879 records to grid cells
Cells with data: 54,947


In [11]:
# ── Compute Shannon entropy per grid cell ─────────────

def shannon_entropy(proportions):
    """Shannon entropy from a list of proportions (0-1)."""
    proportions = proportions[proportions > 0]
    if len(proportions) == 0:
        return 0.0
    return -np.sum(proportions * np.log2(proportions))


records = []

for cell_id, group in df_hcad.groupby("cell_id"):
    total_area = group["lotarea"].sum()
    if total_area == 0:
        records.append({"cell_id": cell_id, "landuse_entropy": 0.0})
        continue
    
    lu_area = group.groupby("landuse")["lotarea"].sum()
    proportions = (lu_area / total_area).values
    
    records.append({
        "cell_id": cell_id,
        "landuse_entropy": round(shannon_entropy(proportions), 4),
    })

df_mix = pd.DataFrame(records)

# Ensure all grid cells are present
df_result = df_grid[["cell_id"]].merge(df_mix, on="cell_id", how="left")
print(f"Computed entropy for {len(df_result)} cells")
print(f"Entropy: mean={df_result['landuse_entropy'].mean():.3f}, "
      f"min={df_result['landuse_entropy'].min():.3f}, "
      f"max={df_result['landuse_entropy'].max():.3f}")

Computed entropy for 54947 cells
Entropy: mean=0.368, min=-0.000, max=3.258


In [12]:
# ── Save output ───────────────────────────────────────
output_path = f"{CSV_DIR}/04_land_use_mix.csv"
df_result.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_result)} rows x {df_result.shape[1]} cols)")
df_result.head(10)

Saved: csv/Houston/04_land_use_mix.csv  (54947 rows x 2 cols)


,cell_id,landuse_entropy
0,cell_00005269,-0.0000
1,cell_00005270,0.3311
2,cell_00005271,0.5792
3,cell_00005946,0.5149
4,cell_00005947,-0.0000
5,cell_00005948,-0.0000
6,cell_00005949,0.1386
7,cell_00005950,-0.0000
8,cell_00005951,0.4176
9,cell_00006625,0.9875


# 04 — Land Use Mix

Computes Shannon entropy of PLUTO land use distribution per grid cell to measure how mixed vs. homogeneous each cell is.

**Data source:** PLUTO CSV (NYC only — `needs_pluto`).

**Note:** Raw area ratios (`comarea`, `resarea`, etc.) are intentionally excluded to prevent Y variable leakage. HHI dropped (r=-0.92 with entropy).

**Output columns:** `cell_id`, `landuse_entropy`

**Output file:** `csv/04_land_use_mix.csv`

In [13]:
# ── Papermill parameters ──────────────────────────────
GRID_CONFIG = "grid.json"

In [14]:
import pandas as pd
import numpy as np
import json
import os
import math

with open(GRID_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

CSV_DIR = config.get("csv_dir", "csv")
os.makedirs(CSV_DIR, exist_ok=True)

if not config["feature_flags"].get("needs_pluto", False):
    print("PLUTO not available — skipping notebook 04.")
    df_grid = pd.read_csv(f"{CSV_DIR}/01_grid_definition.csv", dtype={"cell_id": str})
    df_empty = pd.DataFrame({"cell_id": df_grid["cell_id"]})
    df_empty["landuse_entropy"] = np.nan
    df_empty.to_csv(f"{CSV_DIR}/04_land_use_mix.csv", index=False)
    raise SystemExit("Skipped — needs_pluto=false")

PLUTO_PATH = config["pluto_path"]
BOROUGH_CODES = config["borough_codes"]
BOROUGH_FILTER = config["borough_filter"]
CELL_SIZE_M = config["grid_cell_size_m"]
boro_code_filter = [str(BOROUGH_CODES[b]) for b in BOROUGH_FILTER]
print(f"Loading PLUTO from {PLUTO_PATH}")

PLUTO not available — skipping notebook 04.


SystemExit: Skipped — needs_pluto=false

c:\Users\User\Desktop\00-MaCAD\DataEncoding\OSMnx-data-scraper\venv_data\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [15]:
# ── Load PLUTO + assign to grid cells ─────────────────
COLS = ["borocode", "landuse", "lotarea", "latitude", "longitude"]

df_pluto = pd.read_csv(PLUTO_PATH, usecols=COLS, dtype={"landuse": str})
df_pluto = df_pluto[df_pluto["borocode"].astype(str).isin(boro_code_filter)].copy()
df_pluto["lotarea"] = pd.to_numeric(df_pluto["lotarea"], errors="coerce").fillna(0)
df_pluto["latitude"] = pd.to_numeric(df_pluto["latitude"], errors="coerce")
df_pluto["longitude"] = pd.to_numeric(df_pluto["longitude"], errors="coerce")
df_pluto = df_pluto.dropna(subset=["latitude", "longitude"]).copy()

# Grid parameters — must match notebook 01
REF_LAT = df_pluto["latitude"].mean()
LAT_STEP = CELL_SIZE_M / 111_000
LON_STEP = CELL_SIZE_M / (111_000 * math.cos(math.radians(REF_LAT)))
BUFFER = LAT_STEP
LAT_MIN = df_pluto["latitude"].min() - BUFFER
LON_MIN = df_pluto["longitude"].min() - BUFFER

df_pluto["grid_row"] = ((df_pluto["latitude"] - LAT_MIN) / LAT_STEP).astype(int)
df_pluto["grid_col"] = ((df_pluto["longitude"] - LON_MIN) / LON_STEP).astype(int)
df_pluto["cell_id"] = "r" + df_pluto["grid_row"].astype(str).str.zfill(4) + "_c" + df_pluto["grid_col"].astype(str).str.zfill(4)

# Only keep lots in valid grid cells
df_grid = pd.read_csv(f"{CSV_DIR}/01_grid_definition.csv", dtype={"cell_id": str})
valid_cells = set(df_grid["cell_id"])
df_pluto = df_pluto[df_pluto["cell_id"].isin(valid_cells)].copy()
print(f"PLUTO lots in valid grid cells: {len(df_pluto):,}")
print(f"Unique landuse codes: {df_pluto['landuse'].nunique()}")

NameError: name 'PLUTO_PATH' is not defined

In [16]:
# ── Compute Shannon entropy per grid cell ─────────────

def shannon_entropy(proportions):
    """Shannon entropy from a list of proportions (0-1)."""
    proportions = proportions[proportions > 0]
    if len(proportions) == 0:
        return 0.0
    return -np.sum(proportions * np.log2(proportions))


records = []

for cell_id, group in df_pluto.groupby("cell_id"):
    total_area = group["lotarea"].sum()
    if total_area == 0:
        records.append({"cell_id": cell_id, "landuse_entropy": 0.0})
        continue
    
    lu_area = group.groupby("landuse")["lotarea"].sum()
    proportions = (lu_area / total_area).values
    
    records.append({
        "cell_id": cell_id,
        "landuse_entropy": round(shannon_entropy(proportions), 4),
    })

df_mix = pd.DataFrame(records)

# Ensure all grid cells are present
df_result = df_grid[["cell_id"]].merge(df_mix, on="cell_id", how="left")
print(f"Computed entropy for {len(df_result)} cells")
print(f"Entropy: mean={df_result['landuse_entropy'].mean():.3f}, "
      f"min={df_result['landuse_entropy'].min():.3f}, "
      f"max={df_result['landuse_entropy'].max():.3f}")

NameError: name 'df_pluto' is not defined

In [19]:
# ── Save output ───────────────────────────────────────
output_path = f"{CSV_DIR}/04_land_use_mix.csv"
df_result.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_result)} rows x {df_result.shape[1]} cols)")
df_result.head(10)

Saved: csv/Houston/04_land_use_mix.csv  (54947 rows x 2 cols)


,cell_id,landuse_entropy
0,cell_00005269,-0.0000
1,cell_00005270,0.3311
2,cell_00005271,0.5792
3,cell_00005946,0.5149
4,cell_00005947,-0.0000
5,cell_00005948,-0.0000
6,cell_00005949,0.1386
7,cell_00005950,-0.0000
8,cell_00005951,0.4176
9,cell_00006625,0.9875
